In [1]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments

from PIL import Image
import pandas as pd

/home/kushal/Projects_WSL/OCR/http/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-09 06:54:31.574952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-09 06:54:31.728348: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752044071.802611    5224 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752044071.823719    5224 cuda_blas.cc:1

In [ ]:
# Load your data
data = pd.read_csv("your_data.csv")


In [ ]:
class HandwrittenOCRDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        img = Image.open(item["image_path"]).convert("RGB")
        return {"image": img, "text": item["text"]}


In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

In [ ]:
def preprocess(examples):
    pixel_values = processor(images=examples["image"], return_tensors="pt").pixel_values[0]
    labels = processor.tokenizer(examples["text"], return_tensors="pt").input_ids[0]
    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
train_dataset = HandwrittenOCRDataset(data)
train_dataset = train_dataset.map(preprocess)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-handwritten-finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    fp16=True,
    save_steps=50,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()